# IFRS 9 Expected Credit Loss (ECL) Model
## Notebook 3 — LGD & EAD Models

This notebook covers the LGD and EAD components:

**Loss Given Default (LGD):**
- Collateral-adjusted LGD estimation
- Product-specific LGD floors (Basel III reference)
- Haircut methodology for collateral liquidation costs

**Exposure at Default (EAD):**
- EAD = Outstanding + Undrawn × CCF
- Credit Conversion Factor (CCF) by product type
- Loan-to-Value (LTV) distribution

_Reference: Basel III (BIS 2017) | IFRS 9.B5.5.29_

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import generate_loan_portfolio, assign_stage
from src.lgd_ead import compute_ead, compute_lgd, lgd_ead_summary, CCF_BY_PRODUCT, BASE_LGD

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df = assign_stage(generate_loan_portfolio(n_loans=5000, random_state=42))
df['ead'] = compute_ead(df)
df['lgd'] = compute_lgd(df, collateral_haircut=0.20)

print(f'Portfolio EAD: THB {df["ead"].sum()/1e9:.2f}B')
print(f'Portfolio avg LGD: {df["lgd"].mean():.2%}')
df[['outstanding_balance','undrawn_commitment','ead','lgd','loan_to_value']].describe().round(4)

## 1. Credit Conversion Factor (CCF) by Product

In [ ]:
ccf_df = pd.DataFrame([
    {'product': p, 'ccf': ccf, 'base_lgd': BASE_LGD.get(p, 0.45)}
    for p, ccf in CCF_BY_PRODUCT.items()
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2980b9','#e67e22','#27ae60','#e74c3c','#8e44ad']

axes[0].barh(ccf_df['product'], ccf_df['ccf'] * 100, color=colors, alpha=0.85)
axes[0].set_xlabel('CCF (%)')
axes[0].set_title('Credit Conversion Factor (CCF) by Product')
for i, (v, p) in enumerate(zip(ccf_df['ccf'], ccf_df['product'])):
    axes[0].text(v * 100 + 0.5, i, f'{v:.0%}', va='center')

axes[1].barh(ccf_df['product'], ccf_df['base_lgd'] * 100, color=colors, alpha=0.85)
axes[1].set_xlabel('Base LGD (%)')
axes[1].set_title('Unsecured LGD Floor by Product (Pre-Collateral)')
for i, v in enumerate(ccf_df['base_lgd']):
    axes[1].text(v * 100 + 0.5, i, f'{v:.0%}', va='center')

plt.suptitle('LGD & CCF Parameters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/03_lgd_ccf_parameters.png', bbox_inches='tight')
plt.show()

## 2. LGD Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['lgd'], bins=40, color='#e74c3c', alpha=0.8, edgecolor='white')
axes[0].axvline(df['lgd'].mean(), color='black', linestyle='--', lw=2,
                label=f'Mean LGD = {df["lgd"].mean():.2%}')
axes[0].set_title('LGD Distribution (Portfolio)')
axes[0].set_xlabel('LGD')
axes[0].set_ylabel('Count')
axes[0].legend()

lgd_by_product = df.groupby('product_type')['lgd'].mean().sort_values()
lgd_by_product.plot(kind='barh', ax=axes[1], color='#e74c3c', alpha=0.85)
axes[1].set_title('Average LGD by Product Type')
axes[1].set_xlabel('Average LGD')

plt.suptitle('Loss Given Default (LGD) Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/03_lgd_distribution.png', bbox_inches='tight')
plt.show()

print('=== LGD & EAD Summary by Product ===')
print(lgd_ead_summary(df).to_string())

## 3. LGD vs Loan-to-Value Ratio

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df.sample(min(2000, len(df)), random_state=42)
axes[0].scatter(sample['loan_to_value'].clip(0, 3), sample['lgd'],
                alpha=0.3, color='#2980b9', s=15)
axes[0].set_xlabel('Loan-to-Value Ratio')
axes[0].set_ylabel('LGD')
axes[0].set_title('LGD vs LTV Ratio')
axes[0].axvline(1.0, color='red', linestyle='--', lw=1, label='LTV = 1.0 (Underwater)')
axes[0].legend()

ltv_bins = [0, 0.5, 0.75, 1.0, 1.25, 1.5, 5.0]
ltv_labels = ['0–50%','50–75%','75–100%','100–125%','125–150%','150%+']
df['ltv_bucket'] = pd.cut(df['loan_to_value'], bins=ltv_bins, labels=ltv_labels)
lgd_ltv = df.groupby('ltv_bucket', observed=True)['lgd'].mean()

colors_ltv = ['#27ae60','#2ecc71','#f1c40f','#e67e22','#e74c3c','#c0392b']
lgd_ltv.plot(kind='bar', ax=axes[1], color=colors_ltv, alpha=0.85)
axes[1].set_title('Average LGD by LTV Bucket')
axes[1].set_ylabel('Average LGD')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('LGD vs Collateral Coverage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/03_lgd_vs_ltv.png', bbox_inches='tight')
plt.show()